# Tutorial 3: Tensor fundamentals

Explore tensor shapes, layout, allocation, execution ordering, and numerical comparisons through libttl's C interface. The executable cells perform CPU calculations and syntax checks; the device examples are source-reading exercises. A CPU runtime is sufficient.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jianwen-zhu/2026-fall-ece467-jzhu/blob/gh-pages/tutorial3.ipynb)

## Setup

Run these setup cells once, then work through the examples in order. In Colab, use a CPU runtime. On your own computer, install Make and a C compiler first.

In [ ]:
%%bash
set -euo pipefail
missing=0
for tool in make cc; do command -v "$tool" >/dev/null || missing=1; done
if python3 -c 'import google.colab' 2>/dev/null; then
    if [ "$missing" -eq 1 ]; then
        apt-get -qq update
        apt-get -qq install -y build-essential
    fi
fi
for tool in make cc; do command -v "$tool"; done


In [ ]:
#@title Restore companion files
import base64, io, zipfile
from pathlib import Path

payload = "UEsDBBQAAAAIAAAAcl2KHdFRgAIAADkEAAAHAAAATElDRU5TRV1SW2/aMBR+96844qmVsm6qpk3am0lMsZbEyDFlPIbEEE8hRrFZxb/fOYG26ySkyOfy3Q6FNJC7xg7BMpb602V0hy7CXXMPj18ev4Ft7Ndv3z/19S5A44c4ut05+jEwtrLj0YXg/AAuQGdHu7vAYayHaNsE9qO14PfQdPV4sAlED/VwgZMdAy74Xazd4IYD1Ih6ujCcjB3CBL+PL/VocbiFOgTfuBrxoPXN+WiHWEfi27veBriLnYVZdduY3U8kra175gag3msLXlzs/DnCaAMaaAgjATc0/bklDa/t3h3djYHWpyQCQ9BzQAekM4Gjb92evnaydTrvehe6BFoXrtlgMVBxijQhH5/9CMH2PUMEh7onr+/qphmSfqJA4y2iQJWXzh8/OnGB7c/jgJR22mk9RjYx/rZNpAqN733f+xeyhidrHTkKPxgz2Kp3/o+dvFwPPfiIUq8S6ACn96veWqGr+x529hYY8mK89T92RqIPEQ/v6h5Ofpz4/rf5gPxLAZVamA3XAmQFK62eZSYymPEK37MENtIs1doATmhemi2oBfByCz9lmSUgfq20qCpQmslilUuBNVmm+TqT5RPMca9U+G+WhTQIahQQ4Q1KiorACqHTJT75XObSbBO2kKYkzIXSwGHFtZHpOucaVmu9UpVA+gxhS1kuNLKIQpTmAVmxBuIZH1AteZ4TFeNrVK9JH6RqtdXyaWlgqfJMYHEuUBmf5+JKhabSnMsigYwX/ElMWwpRNKOxqzrYLAWViI/jLzVSlWQjVaXR+EzQpTZvqxtZiQS4lhUFstCqSBjFiRtqAsG9UlxRKGr4cBEcofe6Em+AkAmeI1ZFy2TxdfiB/QVQSwMEFAAAAAgAAAByXbdVuEqACAAAmxAAABYAAABUSElSRF9QQVJUWV9OT1RJQ0VTLm1kfVfZctvIFX3HV9yyH8Z2QdxESlZc8wCTkIQJtyKh8TgvwybQFHuMLWhAMvOUf8gf5ktybjcASs4kVS6Z3ei+2zl36bcUHlUZXxSirE6U5ZWKpHact28p+mW7WjpOeJQ0paLM/5BRRbrCOVlq2tdZnEh7iIa9697wxqVpXpxK9Xis6F30nkaDwc2///mv0WB4TTPxJOlOpPtEOiKLm3tRnlWl2tdVXmqXIFGWVEHfIggpgSGZlj1iA6I8LRJZycZAUppU5ux0XRR5WfWjP3Se9efB1F9u/R2920V5XWrZb6zu/49zKoM2iIqOMvqW19X7nhNUmgqVZTImDRlQxdYeFFw9Cn2UmkQpKc6jOpVZhVOQ8YMZG9+bLfxeGu96CB5bqolPJArHjdsqO5RCV2UdVXUpXXYKga1jSDSeQluJH7HsGSDCcE6JyB5r8SiplAdZyiySFhkLQ0y79kC/qpKLpEwvhr0B20DvbFh/DEnjvWzcdLoYsLn7E/2iRPYsM/rbsTYhKGUlVAZnECAtwRYBNOoCXkiRwtgWeJAj1Q5HpbOoiXcPlu16FFSsgV2O8ie4ErM2A0Mpi1wrcOH0k35FAYNKA0cpn5RWeWaMkjEfVyIBhNAm/wyczoyXuJjYRTU7EZP32/pr/85fLIzMvKggHiJFVUEKfvfv1uHFCMKKuqIUV1ijS0LrPFIswXmUmbSyOnhgSqUOIqpAbBYrYlG8VnZQ3xl+MCrROcGNR5VxUEFsxoVRb8Dq0U5+F5wBuh8s1w/htsdM20FblJexRvgEopoyCW2+aJHKBjKYx6KaxBGGh/oTtERJHavsEWgm0AHTOosRQmkhahCAIeoJtjkdOxs08hJa2kNstTUCwto4NVkNOE3UAXNbSpD8GkBpc6+o94lCfoHJFRQwaD//POgNe8PLHRUi+sbc5zjWWhoSnk1peBxiMcc1pMpe2yg0MXda7vOX/qNM0/7g8vdWjdnoFaddR3GWVkkRHU102jSl/OB0Op6saeDpGRkjuFn93gl9PnLtMPRkcbECNZm/2nWQbcyouDoVXAREigOWLNZhLJsC0LlmI+04nB3zdvG66vLhCy/AzQ8fZnXJSk2AgUaOSlbmKc1kJNM9Qjd0UaRHY5eqnBYC67HZmLg2IVuklHZ0vTc/cVDEyDqbIlGeIKY5qM8ZaVKfnlV1pIWKylznhwotoSyaA70PHxxnjUNKmwyGhiOSBen/WArOVxfmSQ40x6t8lMYukZ3YePCd8j0DxB4JU3EcnDSGsqZnzn2TaV1edqXA2sdVXNM7jsabbXPjzXujJJYicRoitZ+MJ8AIvAAJVMQy3Bd5035OVKoaDXzdIKEdCAVVXWOnS2keqwP/L41bDdtd0EHbBohNzZsGUkOCPnILqZmA6AUyloyvZ+ssUaCl4IBWTYg07zwfgfErTwDgoS4zm2Dsbo6QGY0tqHz8ADTzZ3YNqWkx1n+xpRIgP8kXVb6tJmyCYcoZ1eaTPookob1sAmaLsXjhTsnqMU6gxIJJXDVY349uctG492m7ug2/eBufgi2tN6tfg5k/ozfeFus3Ln0JwvvVQ0g4sfGW4Vda3ZK3/Ep/DZYzl/zf1ht/u6XVxgkW63ngYy9YTucPs2B5R59xb7lCNgXIKQgNV8QKG1GBv2VhC38zvcfS+xzMg/Cr69wG4ZJl3q425NHa24TB9GHubWj9sFmvtj7UzyB2GSxvN9DiL/xliNa3xB75v2JB23tvPmdVjvcA6zdsH01X66+b4O4+pPvVfOZj87MPy7zPc9+qglPTuRcsXJp5C+/ON7dWkLJx+Ji1jr7c+7zF+jz8m4YBhg64MV0tww2WLrzchN3VL8HWd8nbBFsOyO1mtXAdDidurIwQ3Fv6VgqHml4hgiO8ftj6nUCa+d4csrZ8mV1sD9tyliRpLzLc8VV2QnM9d01MnjvToiKTULqPs9GOK4XISH7nD8x1kMXct303Uf+w6QfuWNnoDcgJZ3cYypGI5NVkOL6Z7K/HQo5G8XgSD69Hl+Px5SEeDGQUXwozlmiarh/I9vpEnFAeD3Vmsh61mgeeFANoyZ+Q9ShVeQmCcq3WaPF7VOwXSy8W6ReMR7HgzH7h0UWS54WTl48ia6y2iWzGryJXZgLMnuzgYYfNdnKIFdp9lZx61kk7Izlt8zXzk+33hLFqD9mpmX/+K5zdPGbEtMNQN8rtzJka7VH3jju3WaNAxy+W8EskuUBnN5v46NgPVf5NwjOzT81cjCrPJ7VtPmZWOU9x5XlOsaPCnxDA/tnZpm4wVxGXVJmgQsJkDI4JYNZNiBoMI1Qb7bRVuuvAXF2s6VyDOnvJWwfN2ICZtI54Eonlk/G0zyMIuhrKaAuGcaXFgc9oWdFLWJuwdtMkcwvth9vYq8EnPxzAaRywzG3eVxc8ATR+DEfjhXPmh3u22VInRBJtj+Kb1IXkcJrPXH8xGRqHVNnFWxuoUHsuRpMrehJJ3bxo7JvnkzHpPAeWMgUyTCP5HVaxHzCwko1zWnKjcEkqdugCI6wduy00ZlJrM7zNyMGN2F9eDa7jKBLyejgZy49yH+1vxns5mYxvZHQZjybXV8h5Yd8amCOPsAI9Bcq7OsEvqxysl3+vLU/oWXKfA4EKmfFDKjlxvUDr1rb1TJtY2UnqU2vYT9pZn8CQjCfXRJ6Hhe6NUigjwPTIV4+UHxE2OPBzRPLkZJ9IZijgUYYpLzCpC9IpN8ZzPDmBebbclXkGaiexyPovRO067FDOJuNocHOIR5eXw4m4OtxEH/Enjj4ODtejSSyv44/YuxpzprwwRukXc0abZQz0dDb3LkAdHhL5vXh+c6+N/WZYYDLZyen/sCFRe7zuTIgZNH6dMMu7OQHY8N0mz5tBq3lnI80K5LmRsONBE/SpE0j9D1BLAwQUAAAACAAAAHJdLJT6fagDAACNCQAAJQAAAGNvdXJzZS9wcmVsYWJzLzAxX2R0eXBlc19hbmRfbGF5b3V0LmOtVW1vm0gQ/u5fMUflChKSAOEcR64juSnVRedcotiV7uRDiJgl3isBC5Ymaev/3tldXhbqnO7DIUvg2dlnZp55ZvcNTddJGRHQtjlJwvvjjTYYvKmN72jK2MuWFMebi8EA/8BjSFP9S0YjA74NAJ91lhYMGEuCiHsGDMS7WPkwrVz4s1zOgw/Lv2694P3NzdxU/l/9sRz3/p865p6NH+c3s6U9Mvsm9O6b3MBzru1XQcaB516f/rQLzeNrS+7aTZTqCvqVF0aet2TNSBTcU1bVh5njj2fAE+NvV1jG/77/hZEiiLM8KMgXkkqoMxPw5+Bm25VvV1jOEEpgbUtW6BrmDEn2QNdhUjENYRrBNlx/JhEULMvDB6IZMjyGAL2KTjGINcHXO5FPFutyuwEnXcPK8o0JHB5SQ+lfpxBeP6K1TUdD8EQjtqkhqF9lwJ96F69aJNHpy+Vv3uXveotVlSBJavEEF2+F0VCwaQy6SOeXaa8/1Ifv36ugncUe+ZiqUqdgbZuj1GO9YBHJcxO0Mq131wQLZofF36lmdrbWT1tOGj4ShRUldf7khJV5Ct6fV8vg4+xq/unOax12zVeVkAYwPLLdAobO11J0AVkBkpBHkiIFRxcw5HZe4L7MXk3KrLAkuzKBnRTdyQHMePMZfSizsoAVShwnx/Uhz56OHsN/kAdG0gJfm7Coc0Gachoh8StbjoTtH8PBiTISWM/IRVEUm3BL5ABUyJ3JadwEnvSrIfc60jQiz5UfThFC7nWr8gyyOC4Ig2mrJ7Hf8uGgjonfh5XZVsx2a3YUs+PvC8eJbWL1gx80wfXK39D3jdZPJyGf3HHVLj4HPVwUvXNaz4BidEeq4GN5rCThS1YywFNlXSYho1kKcUgTEnEhgZwERbuv6lZqttar6O50NdTg9u4qGrmgma98+9DVq9SFhWMvv+zmy6mPliaIYP6/RemPRdNjs+2r2fayF0m0e7qyTcc89fm4dRmfNpHUEJrK/1RJRpwhcZKFzB718+oCmyqEMWkmk19yqDQxaHj9oOCPrONfTUDJhjmB1ZmYgHNuMPHSeIKU3t8neHjRvGC9gSyxxDFKVV4kQbx15RRZzw7CWM/W+a5Vmq54YbC3fD02uLjOuOB6yxcX4IpFBxdbsStOdgfjvI9hKxjWHvEKEl2ROk0f/jcRq9jIr20Kdi3R+eq+ldfLSpDEOfLrm1duvr3z5rP3YNlwO1ssvA/1qhp58eny0lssJoPd4AdQSwMEFAAAAAgAAAByXQnB1gArAwAAbggAABsAAABjb3Vyc2UvcHJlbGFicy8wMl90ZW5zb3JzLmOVVW1P2zAQ/t5fcQoCJSOwtqAhrZSJl6KhVQPRIm1iVeQmDrVI48hxYAXx33d+yRtCZURVkzvfc/Y9d77bYGmYFBEFJxM0IfPdhdPpbJTKwyWRi93FUUOTy2jOeaKUHZZKWBKWug+cRR48dwCfnD3RQEJEH1hIg5AXaDSE7kAvTqfj4PT76PSHK2USNG3crabkecaexeC2PaGrcif1xFkhc9dJOZAwpAkVRHIBLAfyQBjGk1CIUSEXFHJcDiWN1CFgTsJ7mkZ/UscHDIkKYXdUj6CyECmMfl1Mg/Pji/HN9cgsvnT0q3F2CZ/MF8b482Y8HlQGkqY5F8rAfLUM3uSBZzR1uz5YJhQH2jjkaS4Byf6yj+7yBcno7QzdPe/5sP9iHMYJJ8oG6bjt9WdNJS+k1SpMtwTgiVybK6YThK9D6PUHsL3NPOuKKYyr/XgMPsPBbje2p2qHYKOly0yubMZ8bXI2/X01Cs7Hl8fTvb4P+NMBYJQGUmb6TXePgknqGsE3R/J1gfHY1ZJXkWRDESS9r8vNKucrSfNaWzIp+KPW+shwUixTK6A6yKVgEW0uNlR1jiO5ynSxqzcu1QGfXF7aVKvbggUZLHiuLoIUBQaPMmZVsruCF2rbmCQ5XUOECqviYUtJa3mjf/FDVgBVUyrYj2B6iLG0rIUZWl5vZdUfQdYb/g9YM14zosW1AFUDgaqHGqTLYi3Ipq2GWMV7oDq3LWitrso2E1iNsevYHqEyO9x8KswlGd5uJknkq78ZGFKaKtW8qp6l2haCfXATnt6B+vNUzlsKm9AWqm1vuX8DVWZl0Dq3Jn64mZs7po+eJSSkS6wlpa5DRun1gesrlJIldU0SfePKr+7MN3DU24Gv4JjO0vbSvktovaK5Nk65461rVoKSqMqP6ZJVdzGiVyb6vW5ZDyQ1sWIyz22DUh10p2zBbObBEfTozkHchJg5Zgg1s8gHR+C4i7BbsAwiFsdU4EiTgCNMMQvIsx5drDG11k4uM71ezbDyLlFMLV9ZKrxGezNzqVy3Y8kWgJ67V9ej8fEJdPtwdTyZjM4cu9o8x+Tm9HQ0mQw6L51/UEsDBBQAAAAIAAAAcl0R3x0XUgMAANcJAAAbAAAAY291cnNlL3ByZWxhYnMvMDNfc3RyZWFtcy5jrVVtT9swEP7eX3ELAiWQdi0gkChD6ljR0KoN0SJtYqhKE2e1ltiV7RSVl/8+vyV1aQurtnxIYvtefPfcPbeFSZwVCQJvwlAWjRpjr1bbKjdP80iMG+OzWg0TAXmEiT+lOAngsQby4fgBDQUkaIpjNIxpIYU+QLOtDweD3vD8c/f8iy9ENnRl/B13FQRGHqfgL1qSpkpP6kknheC+RyhEcYwyxCJBGWAO0TTC8uoZglRuiDECLo9jgRJ1CRhF8W9Ekp/EC4GLBDFmPaqHIVEwAt3vl4PhReeyd3PdNYfPNf1x7i5g1/zJGL/e9HrtSkAgwilTApwWrBIIYTfHSZI56wRxgUkkMCVLVrhgKMqVlQmjSREjNteLKeFFXu04rrHcVTr6Z+F4JQB0gojfDMFCoJKvhZUDARLlo0NpjY+jCbq9k+YeW839w2djMM1opGQkDrdq+87dpoWo9pVes1SSkPi2ULCuDvk5BSXXhr09HFiDWGn52laA4T20jhvN1F5uMRKbbZRPxMxWTKhFPg1+XHWHF71vncHBfgit0MQhgzWwlJX2z+YMqv/NnFMUr9q8Z1gg38QSmrSFugdp6utVsFrdFlYs31K/vNNOWWUbKZWFuOaiuhpf6ujdqtJWhhbTycw3aZUh2Qj/6oL3ERZ+eau3dFxvTtZDKF2/Hp51yWckHjNKZOb9N/Jh/EmtZNGf6ZYKPbMMqhy93TRzWlS8mUYjbmtANVK97EZ8F8AZtFD9OHVVDJsy2e2pbxgxBM8EV6dMriVzqhxBgtMUMcmwAiSj5kiy8vZDoZkUOyT6KpEaMp2/DYuXOJ0YR5Y362cWCc9aN8Jlkk9AoV1h3Da6lmOlrpPiNjgoeWVeE1qoMaHUhjnOMsyRtJ1wleLGyrllKnqEfsnRp/83KbHNCtq4kpPqpSPddi9u7E7Nd5inmChuWBaDp6cV4Z6qcFfMVlMDYGaJQVSWQkQk2UyjDCeQFEynd+NhqsE0Fedp8wr4fJIhNaWreafxxAS2G0cp5Fx7WY6pPR/NJmcKd0ZnJm/B0kQtj6teXStRAbQ020sJl6vXCtn5sPbcjqP5uR3Ocyd6NrstcHXd7XU+QvMArjr9fvdT2SBuyvs35+fdfr9de679AVBLAwQUAAAACAAAAHJdrZQUXD8EAABADwAAHwAAAGNvdXJzZS9wcmVsYWJzLzA0X2NvcnJlY3RuZXNzLmO9V21v6jYU/s6vOGValVYZN0DK0Kp+6Hq5ulejW3Wh0qauitLEFGshiWKH0tvb/75jOy82pOGyTeODwfbj8+ZzHh++o3EQ5SGBbpqRyH/oLbudDo05rHwaW+uEhifw0gH8MPqFeBxCsqYB8YIkR9AFOOdycz6felcfJ1e/WJxHno6xjvXZyYnC0wVYpiQUVWoSn0Wac2Z14wT8ICARyXyeZEAZ+Gufop0RgQUu8CUBhtsBJ6EwAh784C8Sh3/GXRsYD0mWFRrFJyM8z2KY/P5p7n24/DS9/TxRm68d+RUkMeOA3o9c9JQt/ZTc3aOTL+7ruQZYRInP0S6e+5G39qOcMAXr95yFDQM5DnuO08dvF2cNpzOyIBmJ0f8WAeZxLbIcTtUvPPTr7XRaAziJWZIJgLKvBVCZ0IJJRThDb5G6LSDiroYeC/yoTRIZr5y9ILnvNRvWmGJJSmLLsaFIsjK9TGwhn6xS/lxknS0h7+d/3Ey8D9PfLufDgQ19W905ilPB2y/OWyaMW1WCtQut/GqV+5RRTixlgG1mmS2LMFlYxurJN4jTSqCwwd5JwUr69oZUUN1WECVM3BHLI453xunjUhDBi6Ol6RYmSnCqY0xLJbryWLOwT34YleOxVNTsapuAYSVAWqETkJTYo0ydh69flTO9FWUrnwfLgpqOLqCPm1UIFegpyRj3IhoTP/NoHJKNAA6ElCOpqRLcQGt5TDapIq0gWaV+RlkSg4rXwdQlxjRD1lpYXWnbTyD1XiB5lq4QdvH9lxxJfeP5D/i7N3oEn4s1oa7SUnlnhsAuV9XxJMo58dC+JGs4uhuYwo3SRhmd0sZnwkyrxARfIjnZtk3FtcEKbUe8Ypyui50ycd+dwmRw3QdMiySk8SODOyxMLE9MD/ce/IzAHc5cG85sGN0LgU8Q0wf5zlB0qAen7zQSz9GVMWZ2wY86hzsbFyU5m9FZY7IbZFQ59wYruZ6w2iCRmpEPKfv6lG3aXBW9sVpRSkMxq7egtZR1bVo1Oj2nGI4180UlHuG8rV6UQPV0uhgsdYkQ0hDiRPQqmKpqdzioNf6zSpIKC00yZwrlKjJKF05F57GjsVvm21YnIZ81rZ8YGh1BmUzyHTVTaSgithmOxeg6zafEw2o0EQNBmoMfxTBuaD2UNYae3hm2GnXzsTdx25+9sTdxr4cqa2vP8dbrRuHfvqyoYnztNKmo2oz/6vE2xG81KIeUYO28rd90VYDa2kEveu2xredCLbdeO0julqu2kTeVdH2xjTWEd3u6BGHoPmbRg7hjoMYvAtfWK+hBa5WDOIOohGCjZTgSkDbukkU3Vlr+D+oStQd+HIKokELtQdylpURIGM+SZ2s77Xd69xKold/bmJoF3sRor9ybmG8wp2jjd/4/lfvFv4ZzLXw3nyfTy5/BceHmcjabvO8Wu3rsZ7dXV5PZ7Lzz2vkbUEsDBBQAAAAIAAAAcl1/oEXP/QAAAL0CAAAXAAAAY291cnNlL3ByZWxhYnMvcHJlbGFiLmidktFLwzAQxt/7V5wrgxSK+KxOqKOysYKyZeDDINTmQg9DMpoMB7L/3XQT1iJuznsI5OPux/cdF5MyEhVwXoiXeV5kj2ISxUEhg30xislUeiMR7jW9ea+v64eu6Lwk22o9KbQe+jrI8SQfzxhu1w06R9YkcHmtovaVFj7/MXyC2FZljfMQEgrnS79x4vCpaqzevyUYQTfA3RkiKWA/EFej/TYWPOPLhXieJeezHIltqXVDxisW1oxNk8Jg6ECVpFHewtCtzCCF+Ggy3YfQpfMidNuGJcF2H4hb8ix/nXLxlE2L5Tz/JdjfLO4unj1F3MFHHaIBu0nCNaGRpKIvUEsDBBQAAAAIAAAAcl2MXExjegIAADQEAAATAAAAZGVwcy9saWJ0dGwvTElDRU5TRV1SS4/aMBC++1eM9rQrRdtqDz30ZhKzWE3iyDFLOYbEEFchRrEp4t93JrC7bSWkyPP4XkMhDeSutWOwjKX+dJ3coY/w2D7By9eXbzC4XYwDtH6Mk9udo58CY5Wdji4E50dwAXo72d0VDlMzRtslsJ+sBb+Htm+mg00gemjGK5zsFHDB72LjRjceoEHU05XhZOwRJvh9vDSTxeEOmhB86xrEg86356MdYxOJb+8GG+Ax9hYe6vvGw9NM0tlmYG4E6r234OJi788RJhvQQEsYCbixHc4daXhvD+7o7gy0PocQGIKeAzognQkcfef29LWzrdN5N7jQJ9C5cMsGi4GKc5oJ+fjiJwh2GBgiONQ9e/1UN8+Q9BMFGu8RBapcen/814kLbH+eRqS0807nMbKZ8ZdtI1VofO+HwV/IGp6sc+QofGfMYKvZ+d929nK78egjSr1JoAOcPq96b4W+GQbY2XtgyIvxNn/ZmYg+RDy8awY4+Wnm+9/mM/KvBNRqaTZcC5A1VFq9yUxk8MBrfD8ksJFmpdYGcELz0mxBLYGXW/ghyywB8bPSoq5BaSaLKpcCa7JM83Umy1dY4F6p8I8sC2kQ1CggwjuUFDWBFUKnK3zyhcyl2SZsKU1JmEulgUPFtZHpOucaqrWuVC2QPkPYUpZLjSyiEKV5RlasgXjDB9QrnudExfga1WvSB6mqtlq+rgysVJ4JLC4EKuOLXNyo0FSac1kkkPGCv4p5SyGKZjR2UweblaAS8XH8pUaqkmykqjQanwm61OZjdSNrkQDXsqZAlloVCaM4cUPNILhXihsKRQ3/XARH6L2uxQcgZILniFXTMll8H35mfwBQSwMEFAAAAAgAAAByXZZLI0VfAwAA9AUAACIAAABkZXBzL2xpYnR0bC9USElSRF9QQVJUWV9OT1RJQ0VTLm1kdVTBbuM2EL3zKwbJpQXkINk9FO1NseiYrSwZEpM0R1qiLe7KokFSSb1f30fZcRMseuOQ8968eTPSNcnOuHZ2UC4cabDBNNozdn1N37wdGjsg6M0mhJ5Grz2FTlOnVavdzA798ZJFdze/39wSUp1yR9paR3/WZUHg9WbYkRra6YLVTaf3ijKntoG+3H65nd19oVfVm1YFY4cbxub2cHRm1wXK1GB0T2vlvmuH5LuvNJswyLq31geq7Ta8Kacph+7Ba7w/aVS0AwTdIkrH3Yi8u99ClwB6+5WxtXZ746cc49GM05sj7Zwagm4T2jqtyW6p6ZTb6YSChfYjHcAKANqybgdZPya1zG6CMsPUIDXQHZHRIv+uK/atmsbuD2CJea1txr0ewoQH5hX1W9ocWeigpj+38UskuXrv7urXKAP+J+T0wdl2bHBsjT/06jgdgjObMeiE6X90Ew9T4YCm/N6ESdI72fnJ0gFcUSJmaV4h51XTm3Xf/bkH9jMgGhfJ3hfGxIWw9NbZ/acS0dft6AbjOz0hW8u8BU/fkx8333QT4m2EbG3f2zcY8wdjEnFzGf55F8kMn7knMdErmGgQe1ipo6MJUpt+bEHGIkJtYO7F0WnAyQnpdDSsmSZwovsgJNrZ9MrstUtoj+VhG31mRjNQE7uAzNj8edr/OYVnuNHruCg4R5NO9gHE/sfoD/Bx6LX38KjpLiXczwNCLvzsNb6/szv43vaRbq+azgx6dloDtYlKTn43toUJetAOdsWFw8Z6O7pGs14Nu1HtNDbCwnFvHb4vueRUlwv5nFacRE3rqnwSGc/oKq0RXyX0LOSyfJSEjCot5AuVC0qLF/pLFFlC/O91xeuayoqJ1ToXHHeimOePmSge6B64opSUi5WQIJUlxYJnKsHrSLbi1XyJML0XuZAvCVsIWUTORVlRSuu0kmL+mKcVrR+rdVnzhKSQOYeKDOzFTBSLCtX4ihfyBtVxSfwJAauXaZ5PJefl+qUSD0tJyzLPeBUVxzbKglMmalkJaI2SPxlyzyE9vUctaGGx6yxdpQ98QpdIrab3k256XvLpCgrmZSGrdC4htazkJftZQD1LK1HHUouqXCUUvYUL5Tuu4HMp8EuNvn9Wg5QYP9b8QsgynubgqiP4YzJG+y9QSwMEFAAAAAgAAAByXRXqe5YpCAAAERsAABwAAABkZXBzL2xpYnR0bC9pbmNsdWRlL2xpYnR0bC5opRlpb9s49rt/BTH9YhtO2hzbLbadAVRbTb2JDzh2ZwoMINASE3MqkRqSSuLZ6X/fx0OXLSkpJkBimu/ku8m8oncsInfoZvpxvb4JPvdewTfKSLnRe0VZGGcRQR+kiracx6e7X+qbQHK0R5nSe7B5p/kHQZjGmdS/PfKkiGDop/FP6H+9V4RF9K5XyNVCveU0+OKvbqeLeTDz/rtYobOsHWE6B4SLdoSltx5/Rm/aEfz5eDHx+wn+g4sRSijTHylW4W6Afu8h+On3+xkc6OI8UAOLN0AfPqDzywH6G9VgmtjAzt4aWAmyDAdtWrRp5TRoNMuo2RjH28YEg15P7VOivUFYliCl4kAqrDIJbshl3K699eY2WFyjn9Gb0eH2dP7Fu5lOAm91tZn58zUgnR0hLTbrYPEpWHnzKx8Qzo8QNvPbzXK5WK39CcAvjuAfvfG1P58E/moFnv0ZXfa+V5QN1PuGg0R6p3KOyfrr0g8+LhY39YPY/el8/a6ue7F/cV7X2QI+3Sy89dnburoVkKG6bARdBv757AzA/2oEvwv8y9kFgN+2gd/N3gD4384I5qA1G+gQe3sZKAMOcYq3NKZqH0iiDN7rIZrsGU5oiKTiAt8TJHdYkAht94hDJD5y8e1e8CxF/fFm4qFtzMNvIzQjCsdI7QTBkQEPTtHwdS1+x97S+zi9ma6/BpOvc282HQe/LlbXV6vFZhnM/Nli9RX1N2BV0G/cPzOJ8WZgNKJSQblQmg2oBTIStMMsiolEIWYokwRRqEwplAfCVLxHMtyRKItBbYYVfSDoz4xkRHbpNJ1P/CVEEkQqxNbK92a3R+qcGXXWO4K2OPwG0ox4yiDYmKJYEYQR8MbxyTcoWiRGE+8KYWkM5xS5Fzjddekx99bTL35wtfKWnwP/N3+8WeuMP1TlfNDG4Hq++FXn/O11XpBeav2/HcFLTNOG26x+pZyA/7LQxl9EHmhIKksdgg14ijDJRWXZgudCo1y28aMJEeWqBSvhOoIqyxY85+py2YKXCg7OT6rrbsyAPJEwU5Sz5t1aYlfIw5hLEggis1i5Kqc7MaLSgt6bLUn/ghNBB5OJbjZByDOmaiBIdamCGEIMi0Dn15MFRzzbgmES/BTgreRxpkhAhODiCCxIbKI+B38/0q/tDDLEMRZO+0opQ+bTCsqYNo1F0T+ue9rP98X2Xcyxsn/z7e/oAccZyRWywtpUye2+BRNQdu8EhhyyHoVQG9EQ6qVTqRqiaGhXuZQDPnm59bVpkII5BxyE+COztTamWyCCMyqq6yrUD40BisZIxwNsSJDqKq6pJ/n0YIThlAYPREiwUP+B02jwvlfVWKPEGLxrHNOGYW2uT9c/cgGguzApQVuqgkcaqV0jerUrV4hcmwm2e0Vkv9nfozwkSUwSqPA2WIvdoaEduPZlKolE0LQQDkOgEBhEnHAW70/RZw4nzDvbDkoz48iVIdtRjCmPVbXlyYjt51LNt8aDWWwO/ShHTrmkOmdHtWqHhkO77OBSNGmam8e6qc7HrkaNbR00rfAAUdrdVfqIQLjzfb+JpbbqkWouxEmSgowdmPQ5vwnMYEiwiucTyBAGC0BpyBuXOE1GqUp+mTU6FKrQHyjVqlBhOgesmu4w8V1AriBBEcQWehRUTwgQl3LPwp3gjMM4rc13soVQirCAESvV4QqBArPKUs8Yka1clz00NHELkU0eCEO2LOcJIeEApijE/BExutUFWIvkUXSEsqP3O83Nop2acSZjMENFBpST8zvgoBmcaJPBoW0TTjKw1pagv4jgzbnizGBO22+tink0GIMOJc9EWHonz+c23rrmHXr/WIBlrT1EGbbJ90L+IU/3TarXeDWJducoSPMxBCBm1ZVNOiT7XefJ645G7MoNuDEz9bx5HLsIRiAmzYmKJLAsOmTAaWhE/pEMy6JDhknY50VUc3zY2mscnXZ8oJV6serD52IFpqqyAnYxNDPY0KF3MwQ2it5nUB5+gG1J5ArPFyLoHQ1NuJr29x9glEILsNOPRIrbgqB7oq4WIU9SrCsM7Cb5mDFebprz3Ixx7frhUGU4HrXCBbkjgrA8W9zUWEyMiuvGfQgu5s0D8PFYqQXolbNFPk5JXT+f0piGVLmrpDRnd7MATrYUQh+FmQDl1Im7UuiDt4wGLsVD+KuOzNHcksqqUJaFNr6PmFa6a0mp92GKrJahEphfhcN9B+e8CelsaKlVRbdzsGq3ayptYOgZwTLTMYRNOMVEQe+ApCcCgk43Ff2AgGS2TajSIIhBfTs+8ElLZzG3tR+xtLvf6Q6uV41ZZ1C25J6yfo3ELA4d1u4vSwdmrzSOF7Aa5aE9TGgcU0ngSFFtSLNcaoNGja8zvIfs7fPE3VsjKkgIA8NeG0lhyiSykNM/pC7EqaAP+sUixhnEgdvUmYD1cODeN07cq4UbkbFQ9A4Su9k97pYM40rUr14lUqx2o9o9Gvxhl02GdEiC/JnBEaLOAbhkaFc/OgA7+qpxD1k2tmx30U8F0QXzWa2ei9UiYvIHBDCQXVaVdcCqsiV+gd6qrHV0v0GWXTVN4lZn12vkqPo6UIxK+iZ2cMD8Qg1BblZ1SgetUHbUEw/l7yYNAe1AJnjtvAulxDzAuWRAKcSxvurBWK3Dumg6MFaDNczbCkzeRBRf5etPm/lYv1k1B3l+i++I8uJxB7zo1lU35uCqHyskJUW78Ja4O+byssg75lC8UcD8Y5d1F+bwAx82PFGBDYpvTVYoUZvsUWNU5dPhmIMw72bVWt+b/hf1vfwHlP38P1BLAQIUAxQAAAAIAAAAcl2KHdFRgAIAADkEAAAHAAAAAAAAAAAAAACAAQAAAABMSUNFTlNFUEsBAhQDFAAAAAgAAAByXbdVuEqACAAAmxAAABYAAAAAAAAAAAAAAIABpQIAAFRISVJEX1BBUlRZX05PVElDRVMubWRQSwECFAMUAAAACAAAAHJdLJT6fagDAACNCQAAJQAAAAAAAAAAAAAAgAFZCwAAY291cnNlL3ByZWxhYnMvMDFfZHR5cGVzX2FuZF9sYXlvdXQuY1BLAQIUAxQAAAAIAAAAcl0JwdYAKwMAAG4IAAAbAAAAAAAAAAAAAACAAUQPAABjb3Vyc2UvcHJlbGFicy8wMl90ZW5zb3JzLmNQSwECFAMUAAAACAAAAHJdEd8dF1IDAADXCQAAGwAAAAAAAAAAAAAAgAGoEgAAY291cnNlL3ByZWxhYnMvMDNfc3RyZWFtcy5jUEsBAhQDFAAAAAgAAAByXa2UFFw/BAAAQA8AAB8AAAAAAAAAAAAAAIABMxYAAGNvdXJzZS9wcmVsYWJzLzA0X2NvcnJlY3RuZXNzLmNQSwECFAMUAAAACAAAAHJdf6BFz/0AAAC9AgAAFwAAAAAAAAAAAAAAgAGvGgAAY291cnNlL3ByZWxhYnMvcHJlbGFiLmhQSwECFAMUAAAACAAAAHJdjFxMY3oCAAA0BAAAEwAAAAAAAAAAAAAAgAHhGwAAZGVwcy9saWJ0dGwvTElDRU5TRVBLAQIUAxQAAAAIAAAAcl2WSyNFXwMAAPQFAAAiAAAAAAAAAAAAAACAAYweAABkZXBzL2xpYnR0bC9USElSRF9QQVJUWV9OT1RJQ0VTLm1kUEsBAhQDFAAAAAgAAAByXRXqe5YpCAAAERsAABwAAAAAAAAAAAAAAIABKyIAAGRlcHMvbGlidHRsL2luY2x1ZGUvbGlidHRsLmhQSwUGAAAAAAoACgDLAgAAjioAAAAA"
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as files:
    for name in files.namelist():
        path = Path(name)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe companion path")
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(files.read(name))
print("Companion files ready; existing files preserved.")


## Check the examples

The next cell checks the four C examples against libttl's public header. We will then calculate offsets and compare numerical results on the CPU.

In [ ]:
%%bash
set -euo pipefail
command -v cc
command -v make
root="$PWD"
while [ ! -f "$root/course/prelabs/02_tensors.c" ]; do
    if [ "$root" = / ]; then
        echo "Run the companion-file setup cell first." >&2
        exit 1
    fi
    root=$(dirname "$root")
done
mkdir -p .tutorial-3-work
printf '%s\n' "$root" > .tutorial-3-work/course-root
for source in "$root"/course/prelabs/0*.c; do
    cc -std=c11 -Wall -Wextra -Wpedantic -fsyntax-only \
       -I"$root/deps/libttl/include" -I"$root/course/prelabs" "$source"
done
echo 'Four C pre-labs passed local syntax checks (not device execution).'

## Shape and layout

Read the contiguous-layout calculation in `course/prelabs/01_dtypes_and_layout.c`. For shape `[2,3,4]`, row-major **element strides** are `[12,4,1]`. Strides count elements, not bytes.

**Predict:** for index `[1,2,3]`, compute the element offset, then its FP16 byte offset. Predict before running the small local C calculation below.

The same program also checks the finite-value tolerance rule used later: `abs(actual-reference) <= atol + rtol*abs(reference)`. It is a CPU calculation, not an emulation of a device or a replacement implementation of libttl.

In [ ]:
%%writefile .tutorial-3-work/fundamentals.c
#include <math.h>
#include <stdio.h>

int main(void) {
    const int shape[] = {2, 3, 4};
    const int stride[] = {12, 4, 1};
    const int index[] = {1, 2, 3};
    const int offset = index[0]*stride[0] + index[1]*stride[1] + index[2]*stride[2];
    printf("shape=[%d,%d,%d] element_offset=%d fp16_byte_offset=%d\n",
           shape[0], shape[1], shape[2], offset, 2*offset);
    if (offset != 23 || 2*offset != 46) return 1;

    const float actual[] = {1.0f, 2.0f, 3.001f, 4.0f};
    const float reference[] = {1.0f, 2.0f, 3.0f, 4.0f};
    unsigned tight = 0, loose = 0;
    for (unsigned i = 0; i < 4; ++i) {
        double error = fabs((double)actual[i] - reference[i]);
        tight += error > 1e-6 + 1e-6*fabs(reference[i]);
        loose += error > 1e-6 + 1e-3*fabs(reference[i]);
    }
    printf("tight mismatches=%u; loose mismatches=%u\n", tight, loose);
    return tight != 1 || loose != 0;
}


In [ ]:
%%writefile .tutorial-3-work/Makefile
CC = cc
CFLAGS = -std=c11 -O0 -g -Wall -Wextra -Wpedantic
.PHONY: check clean
check: fundamentals
	./fundamentals
fundamentals: fundamentals.c
	$(CC) $(CFLAGS) $< -o $@ -lm
clean:
	$(RM) fundamentals


In [ ]:
%%bash
set -euo pipefail
make -C .tutorial-3-work check

Expected: element offset **23**, FP16 byte offset **46**, one tight-tolerance mismatch and zero loose-tolerance mismatches.

**Modify–diagnose:** use index `[0,1,2]`. Predict offsets **6** and **12**. The unchanged expected-value check fails; explain it before updating that check. Restore the example before Run All.


## Allocate a tensor and transfer data

Open `course/prelabs/02_tensors.c` (the next cell displays it). The tensor is an opaque handle: the API exposes shape and contents without exposing a backend pointer.

Trace five steps: open device, allocate, write host data, read metadata/data, destroy resources. Allocation with `ttl_tensor_empty` does not initialize values. The following write establishes the contents before they are read.

**Predict:** shape `[3,4]`, FP32: rank **2**, strides **[4,1]**, **48 bytes**, device placement, contiguous layout. Which two calls cross the host/device boundary?

In [ ]:
%%bash
set -euo pipefail
root="$(cat .tutorial-3-work/course-root)"
sed -n '1,120p' "$root/course/prelabs/02_tensors.c"


**Short exercise:** predict what changes if the allocation uses `ttl_tensor_empty_host`. It no longer takes the `device` argument; metadata stays the same except placement. Explain the change in the supplied C source. Changing a shape does not itself transfer or initialize data.

## Ordering is part of correctness

Read the central sequence from `03_streams.c`:

```c
ttl_tensor_copy(middle, source, producer);
ttl_stream_wait(consumer, producer);
ttl_tensor_copy(destination, middle, consumer);
ttl_stream_synchronize(consumer);
ttl_tensor_read(destination, output, sizeof(output));
```

The supplied source checks every return status. A stream orders its own queued work. The wait adds a cross-stream dependency; synchronization blocks the host until earlier consumer work, including its accepted dependencies, completes.

**Predict:** draw `source -> middle -> destination -> host read`. Which edge needs the cross-stream wait? Which call blocks the CPU?


**Reason about the code:** suppose the stream wait is replaced with `ttl_stream_synchronize(producer)`. The result remains correct, but the CPU waits before queuing consumer work. Explain how the ordering changes.

## Correctness is a numerical policy

Read the first comparison in `04_correctness.c`. The actual FP32 values are `[1,2,3.001,4]`; the host reference is `[1,2,3,4]`. `ttl_close` takes **relative tolerance first, absolute tolerance second**.

**Predict:** with `(rtol=1e-6, atol=1e-6)`, one mismatch at zero-based index **2**; with `(rtol=1e-3, atol=1e-6)`, it passes. The local calculation above demonstrated why. The max absolute error is approximately 0.001, not necessarily that exact decimal in FP32.


**Short exercise:** suppose the actual values were `[1, 2.001, 3, 4]` with the same reference. Which zero-based index has the largest absolute error? Apply each tolerance rule above to decide whether the comparison passes.

## Review

Explain the lifecycle: allocate, establish contents, order work, synchronize, compare, and release resources. Which steps involve the host, and which involve the device?

For further reading, inspect the supplied stream and numerical-comparison examples. [CloudMake](https://github.com/jianwen-zhu/cloudmake) provides a local-to-cloud workflow for running configured GPU projects. This notebook's calculations and source inspection run directly in Jupyter or Colab.

In [ ]:
%%bash
set -euo pipefail
make -C .tutorial-3-work clean
echo 'Tutorial 3 CPU checks complete.'